# C23 — Laboratório de Inteligência Artificial
## Aula 6: Redes Neurais Convolucionais (CNN)

**Objetivo da aula:** compreender e implementar uma Rede Neural Convolucional (CNN), relacionando convolução, pooling, camadas densas e treinamento com o desempenho em classificação de imagens.

Utilizaremos o **CIFAR-10**, conjunto de imagens coloridas de objetos do cotidiano. O objetivo é classificar cada imagem em uma das dez categorias. O foco não é apenas obter boa acurácia, mas compreender como a rede extrai características e quais decisões influenciam sua generalização.

# 1. Conceitos Fundamentais

## 1.1. Por que utilizar CNNs?

Redes densas (DNNs/MLPs) conectam cada neurônio a todos os valores da camada anterior. São adequadas a muitos problemas tabulares, mas não consideram explicitamente que pixels próximos possuem relações espaciais.

Uma imagem de 28 × 28 pixels possui 784 valores de entrada. Se esses valores forem conectados a uma camada densa com 128 neurônios, cada neurônio terá 784 pesos, um para cada pixel da imagem. Portanto, essa camada precisará de 784 × 128 = 100.352 pesos, sem contar os vieses. Além do custo computacional, essa arquitetura não explora explicitamente a organização espacial da imagem: padrões semelhantes que aparecem em posições diferentes precisam ser aprendidos separadamente.

As CNNs exploram a estrutura espacial por meio de filtros que percorrem a imagem, detectando padrões locais e reutilizando os mesmos pesos em diferentes posições.

## 1.2. Propriedades exploradas pelas CNNs

- **Localidade espacial:** pixels próximos tendem a conter informações relacionadas. Cada neurônio observa uma pequena região, chamada campo receptivo.
- **Estacionariedade:** um mesmo padrão pode aparecer em diferentes posições. O compartilhamento de pesos permite aplicar o mesmo filtro ao longo da imagem.
- **Composição hierárquica:** características simples formam características mais complexas: bordas e linhas formam contornos e texturas que formam partes de objetos.

## 1.3. Arquitetura geral de uma CNN para classificação

Uma arquitetura típica contém:

1. **Entrada:** imagem representada como matriz ou tensor. Imagens em tons de cinza possuem um canal; imagens RGB possuem três.
2. **Convolução:** filtros aprendíveis percorrem a imagem e produzem mapas de características (*feature maps*).
3. **Ativação:** função não linear, frequentemente ReLU, aplicada aos valores da convolução.
4. **Pooling:** reduz a resolução dos mapas. O *max pooling* mantém o maior valor de cada janela.
5. **Flatten:** transforma os mapas multidimensionais em um vetor.
6. **Camadas densas:** combinam as características extraídas para produzir a classificação.
7. **Softmax:** converte os valores de saída em probabilidades associadas às classes.

## 1.4. Como funciona a convolução?

Um filtro, ou *kernel*, é uma matriz de pesos aprendíveis. Em cada posição, seus elementos são multiplicados pelos valores da região correspondente da imagem; os produtos são somados, gerando um valor de saída. O filtro é deslocado ao longo da imagem, formando um mapa de características.

Em imagens coloridas, o filtro também considera os canais de entrada. Uma imagem RGB possui três canais; o filtro combina informações dos três.

Parâmetros importantes:
- **Tamanho do kernel:** dimensão da região observada.
- **Stride:** quantidade de pixels avançada a cada deslocamento.
- **Padding:** valores adicionados nas bordas, influenciando as dimensões da saída.
- **Quantidade de filtros:** determina quantos mapas de características serão produzidos.

## 1.5. Pooling e redução de dimensionalidade

O *max pooling* aplica a operação de máximo em pequenas janelas, por exemplo, 2 × 2. Apenas o maior valor de cada janela segue para a próxima camada. Isso reduz a resolução espacial e o custo das etapas seguintes, mas também descarta informações e reduz a precisão da localização dos padrões.

## 1.6. Treinamento de uma CNN

CNNs são treinadas por retropropagação do erro (*backpropagation*). A rede ajusta pesos dos filtros, vieses e das camadas densas. Em geral, o gradiente descendente é aplicado em mini-batches. Para classificação multiclasse, uma função de perda comum é a entropia cruzada (*cross-entropy*); a saída softmax representa a distribuição de probabilidades prevista.

## 1.7. Como reduzir o sobreajuste?

Redes profundas podem memorizar os exemplos de treinamento. Estratégias para reduzir esse risco:
- **Aumentar a quantidade e diversidade dos dados:** oferece mais exemplos para aprender padrões gerais.
- **Data augmentation:** cria variações plausíveis das imagens de treinamento, como pequenos deslocamentos ou rotações.
- **Dropout:** desativa aleatoriamente parte dos neurônios durante o treinamento.
- **Early stopping:** interrompe o treinamento quando a validação deixa de melhorar.
- **Transfer learning:** aproveitar uma rede previamente treinada e adaptá-la à nova tarefa.

Nenhuma dessas técnicas garante, isoladamente, melhor generalização. Sua adequação deve ser avaliada no problema.

## 1.8. Limitações e boas práticas

- CNNs podem exigir mais poder computacional e tempo de treinamento que modelos tradicionais.
- O desempenho depende da quantidade, diversidade e qualidade das imagens.
- Bom resultado no treinamento não garante bom desempenho em imagens novas.
- O modelo pode aprender vieses ou características acidentais do conjunto de dados.
- Aumentar a profundidade ou a quantidade de filtros não garante melhoria.
- Treinamento, validação e teste devem ser separados corretamente. O teste deve permanecer reservado até a avaliação final.
- Em aplicações médicas, industriais ou de segurança, métricas agregadas não substituem análise de erros, validação externa e avaliação por especialistas.

**Ideia central:** uma CNN não recebe regras explícitas para reconhecer bordas, texturas ou objetos. Ela aprende filtros e combinações de características a partir dos exemplos.

# 2. Prática Guiada

## Classificação de imagens com uma CNN

O CIFAR-10 possui 60.000 imagens coloridas, resolução 32 × 32 pixels e dez categorias de objetos. O conjunto oficial contém 50.000 imagens de treinamento e 10.000 imagens de teste.

Fluxo da prática:

**Carregar dados → inspecionar imagens → preparar tensores → compreender convolução e pooling → construir a CNN → treinar → avaliar → investigar erros.**

Utilizaremos TensorFlow/Keras. Execute as células na ordem e observe as dimensões dos dados e as mudanças no desempenho.

## 2.1. Bibliotecas e reprodutibilidade

O TensorFlow disponibiliza o CIFAR-10 diretamente. Não é necessário baixar arquivos manualmente.

In [ ]:
# Importação das bibliotecas utilizadas na prática.
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf

from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay

# Fixamos as sementes para tornar os experimentos mais reprodutíveis.
SEED = 42
np.random.seed(SEED)
tf.keras.utils.set_random_seed(SEED)

print("TensorFlow:", tf.__version__)
print("Dispositivos:", tf.config.list_physical_devices())

## 2.2. Carregamento e compreensão do conjunto de dados

O CIFAR-10 contém 60.000 imagens para treinamento e 10.000 para teste. Cada imagem possui 28 × 28 pixels.

In [ ]:
# Carrega o conjunto CIFAR-10 diretamente do Keras.
(X_train_original, y_train_original), (X_test, y_test) = (
    tf.keras.datasets.cifar10.load_data()
)

# Os rótulos do CIFAR-10 são retornados como matrizes coluna.
y_train_original = y_train_original.reshape(-1)
y_test = y_test.reshape(-1)

classes = [
    "Avião", "Automóvel", "Pássaro", "Gato", "Veado",
    "Cachorro", "Sapo", "Cavalo", "Navio", "Caminhão"
]

print("Treinamento:", X_train_original.shape)
print("Rótulos:", y_train_original.shape)
print("Teste:", X_test.shape)
print("Quantidade de classes:", len(classes))

 66011136/170498071 ━━━━━━━━━━━━━━━━━━━━ 6:41 4us/step

In [ ]:
# Exibe dez exemplos aleatórios e seus rótulos.
fig, axes = plt.subplots(2, 5, figsize=(12, 5))
indices = np.random.choice(len(X_train_original), size=10, replace=False)

for ax, idx in zip(axes.ravel(), indices):
    ax.imshow(X_train_original[idx], cmap="gray")
    ax.set_title(classes[y_train_original[idx]])
    ax.axis("off")

plt.tight_layout()
plt.show()

Observe as imagens. Algumas categorias possuem aparência semelhante, como gato e cachorro ou automóvel e caminhão. Além disso, as imagens possuem fundos e condições de iluminação variados, o que torna a classificação mais desafiadora.

## 2.3. Preparação das imagens

Os pixels variam de 0 a 255. Vamos normalizá-los para [0, 1] e acrescentar a dimensão do canal. Como são imagens em tons de cinza, cada imagem terá formato **altura × largura × 1**.

Separaremos parte dos dados de treinamento para validação. O teste permanecerá reservado.

In [ ]:
from sklearn.model_selection import train_test_split

# Normaliza os pixels para o intervalo [0, 1].
X_train_completo = X_train_original.astype("float32") / 255.0
X_test = X_test.astype("float32") / 255.0

# Separa treinamento e validação com proporções de classes preservadas.
X_train, X_valid, y_train, y_valid = train_test_split(
    X_train_completo,
    y_train_original,
    test_size=0.20,
    random_state=SEED,
    stratify=y_train_original
)

print("Treinamento:", X_train.shape)
print("Validação:", X_valid.shape)
print("Teste:", X_test.shape)
print("Canais por imagem:", X_train.shape[-1])
print("Intervalo dos pixels:", X_train.min(), "a", X_train.max())

## 2.4. Visualizando uma operação de convolução

Vamos observar uma convolução controlada em uma imagem RGB. Para simplificar a visualização do resultado, utilizaremos um kernel que combina os três canais e evidencia variações horizontais de intensidade.

**Este kernel é apenas ilustrativo. Em uma CNN treinada, os valores dos filtros são aprendidos automaticamente a partir dos dados.**

In [ ]:
# Seleciona uma imagem RGB.
imagem = X_train[0]

# Kernel 3 × 3 × 3: combina os três canais e destaca variações horizontais.
kernel_2d = np.array([
    [-1, -1, -1],
    [ 0,  0,  0],
    [ 1,  1,  1]
], dtype=np.float32)
kernel = np.repeat(kernel_2d[:, :, None], 3, axis=2) / 3.0
kernel = kernel[:, :, :, None]

imagem_tensor = tf.convert_to_tensor(imagem[None, :, :, :])
kernel_tensor = tf.convert_to_tensor(kernel)

resultado = tf.nn.conv2d(
    imagem_tensor, filters=kernel_tensor,
    strides=1, padding="VALID"
).numpy()[0, :, :, 0]

fig, axes = plt.subplots(1, 3, figsize=(12, 4))
axes[0].imshow(imagem)
axes[0].set_title("Imagem RGB")
axes[1].imshow(kernel_2d, cmap="coolwarm")
axes[1].set_title("Padrão do kernel")
axes[2].imshow(resultado, cmap="gray")
axes[2].set_title("Resultado da convolução")

for ax in axes:
    ax.axis("off")
plt.tight_layout()
plt.show()

print("Dimensão da imagem:", imagem.shape)
print("Dimensão do resultado:", resultado.shape)

**Interpretação:** com `padding='VALID'` e `stride=1`, o filtro percorre posições em que cabe integralmente na imagem. Por isso, a saída fica menor. O resultado pode conter valores positivos e negativos, pois ainda não aplicamos uma função de ativação.

## 2.5. Compreendendo o max pooling

O max pooling reduz a resolução selecionando o maior valor de cada janela 2 × 2. Para a visualização, aplicaremos a operação aos três canais RGB.

In [ ]:
# Aplica max pooling 2 × 2 aos três canais RGB.
entrada_pooling = tf.convert_to_tensor(imagem[None, :, :, :])
saida_pooling = tf.nn.max_pool2d(
    entrada_pooling, ksize=2, strides=2, padding="VALID"
).numpy()[0]

fig, axes = plt.subplots(1, 2, figsize=(9, 4))
axes[0].imshow(imagem)
axes[0].set_title(f"Original: {imagem.shape}")
axes[1].imshow(np.clip(saida_pooling, 0, 1))
axes[1].set_title(f"Após max pooling: {saida_pooling.shape}")

for ax in axes:
    ax.axis("off")
plt.tight_layout()
plt.show()

O pooling reduziu altura e largura pela metade. Isso diminui o volume de dados processado nas etapas seguintes, mas também descarta informações espaciais. Portanto, pooling não é uma operação sem perdas.

## 2.6. Construção da arquitetura CNN

A rede terá dois blocos convolucionais. Cada bloco contém convolução, ReLU e max pooling. Depois, utilizaremos `Flatten` e camadas densas para classificar as imagens. A última camada possui dez neurônios e utiliza `softmax`.

In [ ]:
# Define uma CNN compacta para imagens RGB de 32 × 32 pixels.
modelo = tf.keras.Sequential([
    tf.keras.layers.Input(shape=(32, 32, 3)),

    # Primeiro bloco: extrai características locais iniciais.
    tf.keras.layers.Conv2D(32, kernel_size=(3, 3), activation="relu"),
    tf.keras.layers.MaxPooling2D(pool_size=(2, 2)),

    # Segundo bloco: combina características anteriores.
    tf.keras.layers.Conv2D(64, kernel_size=(3, 3), activation="relu"),
    tf.keras.layers.MaxPooling2D(pool_size=(2, 2)),

    # Transforma os mapas multidimensionais em um vetor.
    tf.keras.layers.Flatten(),

    # Combina as características para realizar a classificação.
    tf.keras.layers.Dense(64, activation="relu"),
    tf.keras.layers.Dropout(0.30),

    # Produz uma probabilidade para cada uma das dez classes.
    tf.keras.layers.Dense(10, activation="softmax")
])

modelo.summary()

Analise o resumo: a primeira convolução aprende 32 filtros; a segunda, 64. As camadas de pooling reduzem a resolução, `Flatten` transforma os mapas em vetor e a saída possui dez unidades. Observe a quantidade de parâmetros treináveis.

In [ ]:
# Representação gráfica da arquitetura e das dimensões das camadas.
tf.keras.utils.plot_model(
    modelo, show_shapes=True, show_layer_names=True, dpi=80
)

## 2.7. Configuração do treinamento

Para classificação multiclasse com rótulos inteiros, utilizaremos `sparse_categorical_crossentropy`. O otimizador Adam atualizará os pesos com base nos gradientes calculados durante a retropropagação. O *early stopping* interromperá o treinamento se a perda de validação deixar de melhorar.

In [ ]:
# Configura o processo de aprendizagem.
modelo.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

# Interrompe o treinamento se a validação não melhorar.
early_stopping = tf.keras.callbacks.EarlyStopping(
    monitor="val_loss", patience=2, restore_best_weights=True
)

print("Modelo compilado e pronto para treinamento.")

## 2.8. Treinamento da CNN

O treinamento será realizado em mini-batches de 128 imagens. Cada época representa uma passagem completa pelo conjunto de treinamento. Acompanhe a diferença entre as métricas de treinamento e validação.

In [ ]:
# Treina a CNN e registra as métricas de cada época.
historico = modelo.fit(
    X_train, y_train,
    validation_data=(X_valid, y_valid),
    epochs=10,
    batch_size=128,
    callbacks=[early_stopping],
    verbose=1
)

In [ ]:
# Compara a evolução da perda e da acurácia.
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

axes[0].plot(historico.history["loss"], label="Treinamento")
axes[0].plot(historico.history["val_loss"], label="Validação")
axes[0].set_title("Perda por época")
axes[0].set_xlabel("Época")
axes[0].set_ylabel("Loss")
axes[0].legend()
axes[0].grid(True)

axes[1].plot(historico.history["accuracy"], label="Treinamento")
axes[1].plot(historico.history["val_accuracy"], label="Validação")
axes[1].set_title("Acurácia por época")
axes[1].set_xlabel("Época")
axes[1].set_ylabel("Acurácia")
axes[1].legend()
axes[1].grid(True)

plt.tight_layout()
plt.show()

**Discussão:** observe se as curvas de treinamento e validação evoluem de maneira semelhante. Se a perda de treinamento continuar caindo enquanto a perda de validação aumenta, há indício de sobreajuste. Interprete o comportamento considerando o número de épocas e a complexidade da rede.

## 2.9. Avaliação no conjunto de teste

Somente agora utilizaremos o conjunto de teste, que não participou do ajuste dos pesos nem da escolha da época. A acurácia indica a proporção de classificações corretas, mas não revela sozinha quais classes são confundidas.

In [ ]:
# Avaliação final em dados que não foram usados no treinamento.
test_loss, test_accuracy = modelo.evaluate(X_test, y_test, verbose=0)
print(f"Perda no teste: {test_loss:.4f}")
print(f"Acurácia no teste: {test_accuracy:.4f}")

In [ ]:
# Obtém a classe prevista para cada imagem.
probabilidades = modelo.predict(X_test, verbose=0)
y_pred = np.argmax(probabilidades, axis=1)

# Relatório de precisão, recall e F1-score por classe.
print(classification_report(
    y_test, y_pred, target_names=classes, digits=3
))

In [ ]:
# Linhas: classes reais. Colunas: classes previstas.
matriz = confusion_matrix(y_test, y_pred)
disp = ConfusionMatrixDisplay(
    confusion_matrix=matriz, display_labels=classes
)

fig, ax = plt.subplots(figsize=(10, 10))
disp.plot(ax=ax, cmap="Blues", xticks_rotation=45, colorbar=False)
plt.title("Matriz de confusão — CIFAR-10")
plt.tight_layout()
plt.show()

A matriz de confusão permite identificar quais categorias são mais confundidas. Uma classificação incorreta entre camisa e camiseta, por exemplo, pode ser mais frequente do que entre calça e bolsa. Investigue os erros antes de concluir que o modelo é adequado.

## 2.10. Inspeção de predições

Vamos visualizar exemplos classificados corretamente e incorretamente. A confiança corresponde à maior probabilidade produzida pela softmax; não deve ser interpretada automaticamente como probabilidade calibrada de acerto.

In [ ]:
# Seleciona exemplos em que o modelo acertou e em que errou.
acertos = np.where(y_pred == y_test)[0]
erros = np.where(y_pred != y_test)[0]
indices_exibicao = list(erros[:5]) + list(acertos[:5])

fig, axes = plt.subplots(2, 5, figsize=(14, 6))
for ax, idx in zip(axes.ravel(), indices_exibicao):
    ax.imshow(X_test[idx, :, :, 0], cmap="gray")
    real = classes[y_test[idx]]
    previsto = classes[y_pred[idx]]
    confianca = probabilidades[idx].max()
    status = "Acerto" if y_pred[idx] == y_test[idx] else "Erro"
    ax.set_title(
        f"{status}\nReal: {real}\nPrevisto: {previsto}\nConf.: {confianca:.2f}",
        fontsize=9
    )
    ax.axis("off")

plt.tight_layout()
plt.show()

## 2.11. Visualização dos mapas de características

Uma CNN transforma a imagem em mapas de características ao longo das camadas convolucionais. Vamos observar as ativações produzidas pelo primeiro bloco. Os mapas não são necessariamente interpretáveis como bordas ou formas específicas: a rede pode aprender padrões úteis que não são facilmente reconhecíveis por uma pessoa.

In [ ]:
# Cria um modelo auxiliar que retorna as ativações da primeira convolução.
modelo_ativacoes = tf.keras.Model(
    inputs=modelo.input, outputs=modelo.layers[0].output
)

indice = erros[0] if len(erros) > 0 else 0
imagem_exemplo = X_test[indice:indice+1]
ativacoes = modelo_ativacoes.predict(imagem_exemplo, verbose=0)

fig, axes = plt.subplots(4, 8, figsize=(12, 6))
for i, ax in enumerate(axes.ravel()):
    ax.imshow(ativacoes[0, :, :, i], cmap="viridis")
    ax.set_title(f"Filtro {i+1}", fontsize=8)
    ax.axis("off")

plt.suptitle(
    f"Ativações da primeira convolução — classe real: {classes[y_test[indice]]}",
    y=1.02
)
plt.tight_layout()
plt.show()

Observe que cada filtro produz um mapa diferente. Isso ocorre porque os filtros possuem pesos distintos e respondem de maneiras diferentes aos padrões presentes na imagem.

## 2.12. Experimento: alterando a arquitetura

A arquitetura é uma escolha de projeto. Mais filtros podem aumentar a capacidade de representação, mas também elevam o número de parâmetros e o custo computacional. Vamos criar uma versão compacta para comparar a quantidade de parâmetros. Se houver tempo, treine-a e compare a validação com a arquitetura original.

Não escolha a arquitetura olhando o conjunto de teste. Use a validação para comparação e reserve o teste para a avaliação final.

In [ ]:
# Exemplo de arquitetura alternativa mais compacta.
modelo_compacto = tf.keras.Sequential([
    tf.keras.layers.Input(shape=(32, 32, 3)),
    tf.keras.layers.Conv2D(16, (3, 3), activation="relu"),
    tf.keras.layers.MaxPooling2D((2, 2)),
    tf.keras.layers.Conv2D(32, (3, 3), activation="relu"),
    tf.keras.layers.MaxPooling2D((2, 2)),
    tf.keras.layers.Flatten(),
    tf.keras.layers.Dense(64, activation="relu"),
    tf.keras.layers.Dropout(0.30),
    tf.keras.layers.Dense(10, activation="softmax")
])

modelo_compacto.compile(
    optimizer="adam",
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

print("Parâmetros da CNN original:", modelo.count_params())
print("Parâmetros da CNN compacta:", modelo_compacto.count_params())

In [ ]:
# Experimento opcional: treine a arquitetura compacta.
historico_compacto = modelo_compacto.fit(
    X_train, y_train,
    validation_data=(X_valid, y_valid),
    epochs=5,
    batch_size=128,
    callbacks=[tf.keras.callbacks.EarlyStopping(
        monitor="val_loss", patience=1, restore_best_weights=True
    )],
    verbose=1
)

print("Acurácia de validação da CNN original:",
      max(historico.history["val_accuracy"]))
print("Acurácia de validação da CNN compacta:",
      max(historico_compacto.history["val_accuracy"]))

## Reflexão

Responda às questões em uma célula de Markdown após esta seção. Utilize evidências das saídas, gráficos e experimentos realizados.

1. Por que uma CNN tende a ser mais adequada do que uma MLP tradicional para imagens? Relacione sua resposta à localidade espacial e ao compartilhamento de pesos.
2. Na demonstração de convolução, por que a saída ficou menor que a imagem original com `padding="VALID"`? O que mudaria com `padding="SAME"`?
3. Qual é a diferença entre a função de uma camada convolucional e a de uma camada de pooling? Por que não são operações equivalentes?
4. O que representa um mapa de características (*feature map*)? Por que filtros diferentes podem produzir mapas distintos para a mesma imagem?
5. Por que `Flatten` é utilizada antes das camadas densas? Qual informação estrutural é transformada nesse processo?
6. Como a perda e a acurácia de treinamento se comportaram em relação à validação? Há evidências de sobreajuste? Justifique com os gráficos.
7. Quais classes apresentaram mais confusões na matriz de confusão? O que as imagens dessas classes têm em comum?
8. Por que a acurácia, isoladamente, não é suficiente para avaliar um classificador? Comente precisão, recall e F1-score.
9. O que mudou ao comparar a arquitetura original com a compacta? Considere desempenho, parâmetros e custo computacional.
10. Imagine que a CNN será utilizada em uma aplicação real. Que tipo de erro seria mais crítico e quais dados ou testes adicionais seriam necessários antes de utilizar o modelo?

# 3. Prática Contextualizada

Agora você deverá adaptar o conteúdo desta aula para um **problema real de classificação de imagens**.

A atividade será desenvolvida individualmente, e a **pessoa selecionada apresentará o trabalho na próxima aula**, conforme a organização da disciplina.

## O aluno deverá

- escolher um problema real que possa ser tratado como classificação de imagens;
- encontrar um conjunto de imagens adequado e explicar sua origem, suas classes e limitações;
- justificar o problema e a relevância de uma solução automatizada;
- preparar as imagens para entrada na CNN, explicando redimensionamento, normalização, canais e divisão dos dados;
- implementar e treinar uma CNN, adaptando a arquitetura estudada na prática guiada;
- avaliar o modelo com métricas adequadas e apresentar exemplos de acertos e erros;
- interpretar os resultados, discutir vieses e apontar limitações;
- explicar melhorias que poderiam ser investigadas, como data augmentation, dropout, alteração da arquitetura ou transfer learning.

A apresentação deve demonstrar **compreensão e adaptação**, e não apenas reprodução do notebook desenvolvido em aula.

## Apresentação

A pessoa selecionada apresentará o trabalho na próxima aula. A apresentação deverá ser breve e técnica, mostrando o problema, os dados, as principais decisões de implementação, os resultados e suas limitações. A turma poderá fazer perguntas e oferecer feedback; o apresentador deverá justificar as escolhas com base no código e nas evidências observadas.

**Entrega sugerida:** notebook executável no Google Colab, com as saídas preservadas, fonte do conjunto de dados e respostas às questões de reflexão.
